# 01 - KG-RAG Walkthrough

**Lecture goal:** show how Knowledge Graph RAG turns documents into entities and relationships, then uses that graph to answer questions that require connecting facts.

**Use case:** a small AI-labs corpus containing pages about OpenAI, Anthropic, Claude, Gemini, GPT-4, Transformers, RLHF, and related people.

**Notebook sections:**
1. Build phase - load documents, extract triples, and write the graph to Neo4j
2. Inspect the graph - view identified entities and relationships
3. Query phase - ask KG-RAG a question and see the input/output
4. Example KG-RAG inputs and outputs - compare multi-hop and single-hop questions
5. Discussion - where KG-RAG helps and where it can fail

The saved cell outputs are lecture examples. Your exact entity counts, relationship labels, and answer wording may vary because the extraction and final generation steps use an LLM.


## 1. Build Phase - Documents to Graph

KG-RAG starts like standard RAG: it loads source documents. The difference is that indexing does more than embedding chunks. Here, the pipeline asks an LLM to extract paths such as:

```text
Anthropic -> develops -> Claude
Dario Amodei -> co-founded -> Anthropic
Transformer -> introduced by -> Attention Is All You Need
```

Those extracted entities and relationships are stored in Neo4j as a property graph.


In [1]:
from rag_compare.common import load_corpus
from rag_compare.kg_rag import KGRagPipeline

docs = load_corpus()
print(f"Loaded {len(docs)} source documents")
print("Example document preview:")
print(docs[0].text[:300].replace("\n", " ") + "...")

kg = KGRagPipeline.build(docs)
print("KG-RAG index built and written to Neo4j")


Loaded 18 source documents
Example document preview:
# Large language model  A large language model (LLM) is a language model notable for its ability to achieve general-purpose language generation and other natural language processing tasks...
Parsing nodes: 100%|██████████| 18/18 [00:00<00:00, ...]
Extracting paths from text: 100%|██████████| ...
KG-RAG index built and written to Neo4j


### What happened in section 1?

1. `load_corpus()` loaded text files from `data/raw/`.
2. `KGRagPipeline.build(docs)` configured the LLM and embedding model.
3. LlamaIndex `SchemaLLMPathExtractor` extracted entity-relation-entity paths from the documents.
4. `Neo4jPropertyGraphStore` persisted those graph paths to Neo4j.
5. The resulting `kg` object can now answer questions using the graph index.


## 2. Inspect the Graph - Entities and Relationships

A key advantage of KG-RAG is that the intermediate knowledge representation is inspectable. We can look at the graph before asking questions.

Make sure Neo4j is running:

```bash
docker compose up -d neo4j
```

You can also open Neo4j Browser at http://localhost:7474 and log in with `neo4j` / `ragcompare`.


In [2]:
from neo4j import GraphDatabase
from rag_compare.common import get_settings

cfg = get_settings()

with GraphDatabase.driver(
    cfg.neo4j_uri,
    auth=(cfg.neo4j_username, cfg.neo4j_password),
) as driver:
    with driver.session() as session:
        record = session.run(
            """
            MATCH (n)
            OPTIONAL MATCH ()-[r]->()
            RETURN count(DISTINCT n) AS entities, count(r) AS relationships
            """
        ).single()

print(dict(record))


{'entities': '<varies by run>', 'relationships': '<varies by run>'}


In [3]:
# Show example relationships extracted into Neo4j.
# Property names can vary by LlamaIndex version, so this query checks common fields.
with GraphDatabase.driver(
    cfg.neo4j_uri,
    auth=(cfg.neo4j_username, cfg.neo4j_password),
) as driver:
    with driver.session() as session:
        rows = session.run(
            """
            MATCH (a)-[r]->(b)
            RETURN
              coalesce(a.name, a.id, a.label, toString(id(a))) AS source,
              type(r) AS relationship,
              coalesce(b.name, b.id, b.label, toString(id(b))) AS target
            LIMIT 10
            """
        )
        for row in rows:
            print(f"{row['source']} --{row['relationship']}--> {row['target']}")


OpenAI --DEVELOPS--> GPT-4
Anthropic --DEVELOPS--> Claude
Google DeepMind --DEVELOPS--> Gemini
Dario Amodei --CO_FOUNDED--> Anthropic
Transformer --INTRODUCED_BY--> Attention Is All You Need
Reinforcement Learning from Human Feedback --ABBREVIATED_AS--> RLHF


### Useful Neo4j Browser queries

Preview graph nodes:

```cypher
MATCH (n) RETURN n LIMIT 50
```

Show extracted relationships:

```cypher
MATCH (a)-[r]->(b)
RETURN a, type(r) AS relationship, b
LIMIT 100
```

Count relationship types:

```cypher
MATCH ()-[r]->()
RETURN type(r) AS relationship, count(*) AS count
ORDER BY count DESC
```

Search for one entity:

```cypher
MATCH (n)
WHERE toLower(coalesce(n.name, n.id, n.label, "")) CONTAINS "anthropic"
RETURN n
LIMIT 25
```


## 3. Query Phase - Question to Graph-Grounded Answer

At query time, KG-RAG does three conceptual things:

1. Identify entities in the question.
2. Retrieve a relevant subgraph and linked source text.
3. Ask the LLM to synthesize an answer from that graph-grounded evidence.

The input is just a natural-language question. The output is an answer grounded in the graph and source text.


In [4]:
question = (
    "Which AI lab was founded by former OpenAI employees "
    "and develops the Claude model?"
)

print("KG-RAG input question:")
print(question)

answer = kg.query(question)

print("\nKG-RAG output answer:")
print(answer)


KG-RAG input question:
Which AI lab was founded by former OpenAI employees and develops the Claude model?

KG-RAG output answer:
Anthropic. The graph evidence links Anthropic to former OpenAI employees as founders and also links Anthropic to the Claude language model.


### Why this is a KG-RAG-style question

This question is not just asking for one fact. It requires connecting multiple relationships:

```text
former OpenAI employees -> founded -> Anthropic
Anthropic -> develops -> Claude
```

A graph representation makes those joins easier to inspect and reason about than a flat list of chunks.


## 4. Example KG-RAG Inputs and Outputs

The next cells show two example question types. The first is multi-hop and relationship-heavy. The second is single-hop and could also be answered by normal vector RAG.


In [5]:
examples = [
    {
        "label": "multi-hop relationship question",
        "question": (
            "Which AI lab was founded by former OpenAI employees "
            "and develops the Claude model?"
        ),
    },
    {
        "label": "single-hop factual question",
        "question": "Who is the CEO of OpenAI?",
    },
]

for example in examples:
    print("=" * 80)
    print(f"Example type: {example['label']}")
    print(f"Input: {example['question']}")
    print("Output:")
    print(kg.query(example["question"]))
    print()


Example type: multi-hop relationship question
Input: Which AI lab was founded by former OpenAI employees and develops the Claude model?
Output:
Anthropic. The graph connects Anthropic with former OpenAI employees as founders and connects Anthropic with Claude.

Example type: single-hop factual question
Input: Who is the CEO of OpenAI?
Output:
Sam Altman is identified as the CEO of OpenAI in the retrieved corpus context.



### Input/output summary for lecture

| Stage | Example |
|---|---|
| Source document text | Wikipedia-derived pages about AI labs, models, people, and methods |
| Extracted entity | `Anthropic`, `Claude`, `Dario Amodei`, `OpenAI` |
| Extracted relationship | `Anthropic --DEVELOPS--> Claude` |
| User question | `Which AI lab was founded by former OpenAI employees and develops the Claude model?` |
| Retrieved evidence shape | A small subgraph plus linked text chunks |
| Final answer | `Anthropic` |


## 5. Discussion - Strengths, Failure Modes, and Lecture Prompts

### What KG-RAG is good at

- Questions that require joining facts across entities.
- Domains with important relationships: biomedical, finance, legal, organizational knowledge, research graphs.
- Debugging retrieval because the graph can be inspected directly in Neo4j.

### What can go wrong

- The LLM may miss entities or extract incorrect relationships.
- Relationship labels can be inconsistent across runs.
- Indexing is slower and more expensive than plain vector RAG.
- Exact quote or acronym lookup may be better handled by Hybrid RAG with BM25.

### Questions for students

1. Which parts of the answer came from graph structure rather than plain text similarity?
2. What relationships would you expect to see for `Anthropic`, `OpenAI`, and `Claude`?
3. Which extracted relationships look noisy or too generic?
4. How would the result change if the graph missed the `Anthropic -> develops -> Claude` edge?
5. When would you choose KG-RAG over Hybrid RAG?
